# 08 — IBM Quantum Hardware Runner

This notebook is the controlled hardware-execution layer for the adaptive QEM study. It uses **SamplerV2**, the current IBM Runtime sampler interface, rather than the legacy `backend.run()` path. The current IBM documentation specifies `SamplerV2.run(pubs, shots=...)` and backend execution mode. citeturn0search0turn0search5

**Important:** `RUN_HARDWARE = False` by default. Change it only after verifying authentication, backend availability, circuit measurements, and the planned execution matrix.

In [ ]:
from pathlib import Path
import sys, json, platform
ROOT = Path.cwd()
if not (ROOT / 'execute_ibm_qem.py').exists():
    ROOT = ROOT / 'Adaptive_QEM_IBM' if (ROOT/'Adaptive_QEM_IBM').exists() else ROOT
sys.path.insert(0, str(ROOT))
from execute_ibm_qem import load_service, get_backend, prepare_circuit, submit_sampler_v2, extract_counts, software_versions
print(software_versions())
print(platform.platform())

In [ ]:
BACKEND_NAME = 'ibm_kingston'
SHOTS = 4096
OPTIMIZATION_LEVEL = 3
SEED_TRANSPIILER = 42
RUN_HARDWARE = False
print('Backend:', BACKEND_NAME)
print('Shots:', SHOTS)
print('RUN_HARDWARE:', RUN_HARDWARE)

## 1. Authentication and backend check

Use a previously configured IBM Quantum account. Do not paste or store an API token in the notebook. The runner calls `QiskitRuntimeService()` when no explicit credentials are supplied.

In [ ]:
service = load_service()
backend = get_backend(service, BACKEND_NAME)
print('Backend:', backend.name)
print('Operational:', backend.status().operational)
print('Pending jobs:', backend.status().pending_jobs)

## 2. Load benchmark circuits

Import the same circuit-construction functions used by Notebook 02/04 so the hardware experiment is directly traceable to the ideal/noisy benchmark suite.

In [ ]:
from circuits.bell import bell_phi_plus
from circuits.ghz import create_ghz
from circuits.teleportation import teleportation
from circuits.superdense import superdense_coding
from circuits.qft import qft
from circuits.grover import grover_2qubit
from circuits.qaoa import qaoa_two_node
from circuits.qpe import qpe
from circuits.qrng import qrng

benchmarks = {
    'Bell_Phi_Plus': bell_phi_plus(),
    'GHZ_3': create_ghz(3), 'GHZ_4': create_ghz(4), 'GHZ_5': create_ghz(5),
    'Teleportation': teleportation(),
    'Superdense_00': superdense_coding('00'), 'Superdense_01': superdense_coding('01'),
    'Superdense_10': superdense_coding('10'), 'Superdense_11': superdense_coding('11'),
    'QFT_3': qft(3), 'QFT_4': qft(4), 'Grover_2Q': grover_2qubit(),
    'QAOA_2Q': qaoa_two_node(), 'QPE': qpe(), 'QRNG_4': qrng(4),
}
print('Benchmarks:', len(benchmarks))

In [ ]:
prepared = []
for name, circuit in benchmarks.items():
    tc = prepare_circuit(circuit, backend, OPTIMIZATION_LEVEL, SEED_TRANSPIILER)
    prepared.append((name, tc))
    print(f'{name:20s} depth={tc.depth():4d} size={tc.size():4d} qubits={tc.num_qubits}')

## 3. Safety gate before submission

Every circuit must contain measurements. We also print the number of circuits and total requested shots so accidental large submissions are visible before execution.

In [ ]:
assert all(c.num_clbits > 0 for _, c in prepared), 'At least one circuit has no classical measurement bits.'
print('Circuits to submit:', len(prepared))
print('Requested shots per circuit:', SHOTS)
print('Total requested shots:', len(prepared) * SHOTS)
if not RUN_HARDWARE:
    print('DRY RUN: no IBM job will be submitted.')

## 4. Submit raw hardware baseline

Run this cell only when `RUN_HARDWARE=True`. SamplerV2 supports submitting multiple circuits in one job and specifying shots in `run()`. citeturn0search0turn0search5

In [ ]:
if RUN_HARDWARE:
    names = [n for n, _ in prepared]
    circuits = [c for _, c in prepared]
    job = submit_sampler_v2(backend, circuits, shots=SHOTS)
    print('JOB ID:', job.job_id())
    print('Initial status:', job.status())
else:
    job = None
    print('Skipped: set RUN_HARDWARE=True to submit.')

## 5. Retrieve and save results

Do not overwrite the job ID. It is a primary provenance field for the paper. RuntimeJobV2 exposes job status and result retrieval. citeturn0search7

In [ ]:
if RUN_HARDWARE:
    counts = extract_counts(job)
    output = ROOT / 'data/hardware/ibm_kingston_raw_sampler_v2.json'
    output.parent.mkdir(parents=True, exist_ok=True)
    record = {
        'backend': backend.name,
        'job_id': job.job_id(),
        'status': str(job.status()),
        'shots': SHOTS,
        'circuits': names,
        'counts': counts,
        'software_versions': software_versions(),
    }
    output.write_text(json.dumps(record, indent=2), encoding='utf-8')
    print('Saved:', output)
else:
    print('No results retrieved because no job was submitted.')

## Experimental note

For the paper, keep this raw hardware baseline separate from mitigation experiments. Record backend calibration information as close as possible to each execution and retain all job IDs. The current IBM SamplerV2 interface is the appropriate execution path for modern Runtime clients; IBM's current documentation recommends Qiskit 2.5.2 and qiskit-ibm-runtime 0.47.0 or newer, so the exact installed versions must be recorded before the experiment. citeturn0search5